# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib numpy ipywidgets --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

### 1. Introduction

This notebook explains the motivation behind and the basic principle of OFDM. In wireless communication, there is a special problem when larger data througputs are required and fast signals are transmitted (more bits per second, and hence a wider frequency bandwidth of the signal). Remember that in digital modulation, faster signals mean shorter symbol durations, which requires a higher frequency bandwidth. 

We will discuss in the following that when signals travel through the air, due to multiple paths that a signal can travel, multiple signals, although coming from the same source, hit a receiver antenna with slight delays. These delays create phase shifts and the signals hence interfere with each other. One result is that the channel attenuation is different on different frequencies (referred to as frequency selectivity), making it difficult for a receiver to reconstruct data. In addition, with short consecutive data symbols (say, in the order of microseconds), each symbol might even partially overlap and interfere with the next one, as they might travel for different times (resulting in Inter-Symbol Interference, ISI).

These problems are today addressed by Wi-Fi, 5G, and many other broadband systems with a concept called OFDM. 


#### 1.1 The Problem of Multipath Propagation

The illustrations below show how signals might travel over multiple paths. The effect is more relevant when distances are longer (outdoor cellular with houses, cars, or other reflectors), or many obstacles are in the way (indoor networks with many reflectors and obstacles).

<center>
  <img src="../../Admin/picture/multipath_scenario01.png" 
       alt="Scenario 1"
       style="filter: grayscale(50%);"
       width="30%">
  <img src="../../Admin/picture/multipath_scenario02.png" 
       alt="Scenario 2" 
       style="filter: grayscale(50%);"
       width="30%">
  <p><em>Line-of-sight link with only the direct signal (left), and multipath propagation with both direct and reflected signals (right). At the smartphone, the arriving signals can combine either constructively or destructively, depending on their relative phases (frequency selective fading). A transmitted waveform arrives at the receiver over multiple paths (reflection, diffraction, scattering). Each path has a delay and a complex gain (amplitude + phase).</em></p>
</center>




#### 1.2 Harmonic Wave Example
 The combination of two signals is visualized in the following interactive graph below.

In [ ]:
# -- Multipath addition visualization ---
from ipywidgets import interact, FloatSlider

def multipath_sum(phase2=90.0):
    phase1 = 0.0
    freq = 1.0
    amp1 = 1.0
    amp2 = 0.9
    t = np.linspace(0, 2, 1000)

    sig1 = amp1 * np.cos(2 * np.pi * freq * t + np.deg2rad(phase1))
    sig2 = amp2 * np.cos(2 * np.pi * freq * t + np.deg2rad(phase2))
    sig_total = sig1 + sig2

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, sig_total, linewidth=3, color='lime', label="Received (Sum)")
    ax.plot(t, sig1, '--', color='gold', alpha=0.8, label=f"Path 1 ({amp1:.1f} A, 0°)")
    ax.plot(t, sig2, '--', color='orange', alpha=0.8, label=f"Path 2 ({amp2:.1f} A, {phase2:.0f}°)")

    ax.set_xlabel("Time (normalized)")
    ax.set_ylabel("Amplitude")
    ax.set_title("Multipath Signal Superposition")
    ax.set_ylim(-(amp1 + amp2 + 0.3), amp1 + amp2 + 0.3)
    ax.grid(True, alpha=0.3)
    ax.set_facecolor((0.2, 0.2, 0.3))
    ax.legend(fancybox=True, framealpha=1, facecolor=(0.6, 0.61, 0.8))
    plt.tight_layout()
    plt.show()

interact(multipath_sum, phase2=FloatSlider(value=180.0, min=0, max=360, step=1))


### 1.3 Baseband Impulse Example

Now, instead of a harmonic wave, we think of a single impulse being transmitted.

Compare the images below, in which a baseband impulse is illustrated. When an ideal impulse is transmitted, the impact of multipath propagation becomes easy to illustrate. Comparing the line-of-sight (LOS) reception, top-right image, with the multipath reception image, bottom-left, clearly shows how additional paths arrive later at the receiver (there are around eight peaks instead of one). In the frequency domain, the effect appears even more striking: instead of a flat response with LOS (black line), where all frequencies experience the same attenuation, multipath creates frequency-selective fading, with some frequencies attenuated more than others (red line with many ups and downs). As a result, broadband signals with a single carrier and wider frequency bandwidths cannot be reliably transmitted over such channels without countermeasures.

<center>
  <img src="../../Admin/picture/tx_impulse.png" 
       alt="tx" 
       width="45%">
  <img src="../../Admin/picture/rx_los_only.png" 
       alt="los rx" 
       width="45%">
  <img src="../../Admin/picture/rx_multipath.png" 
       alt="multipath rx" 
       width="45%">
  <img src="../../Admin/picture/rx_freq_compare.png" 
       alt="freq" 
       width="45%">
  <div width="80%">
  <p><em>An impulse signal at the transmitter (top left), the corresponding received signal with line-of-sight only (top right), and with multipath propagation (bottom left). The row of delayed impulses illustrates the delay spread, showing how the signal energy is dispersed over time. The frequency-domain representation (bottom right) highlights the resulting frequency selectivity of the channel. All amplitudes are normalized.</em></p></div>
</center>



### 2. OFDM Model

#### 2.1 Introduction

OFDM stands for Orthogonal Frequency Division Multiplexing. It is a digital modulation technique used in systems such as Wi-Fi, cellular 4G Long Term Evolution (LTE), and the latest cellular standard, 5G. The core idea is to avoid transmitting all data on a single wide carrier with a larger bandwidth. Instead, the available band is divided into many narrow subcarriers, and each subcarrier carries a slower stream of data. They are orthogonal, because they normally do not create interference to each other. Each subcarrier carries a low-rate QAM/QPSK symbol, so each sees (approximately) flat fading even if the overall channel is frequency-selective.

#### 2.2 Mitigate Multipath Fading
Because each subcarrier is narrowband, the system is more robust against multipath reflections, which would otherwise cause significant distortion. Instead of "frequency selective fading" for the whole spectrum, each subcarrier only observes "flat fading".

The spreading of impulses at a receiver is represented as **RMS delay spread** $\tau_{\mathrm{rms}}$, measured in seconds, which is an normalized way to summarize multipath spread in time. A large delay spread implies a small **coherence bandwidth** $B_c$, which is the frequency range over which the channel is approximately flat:

$$
B_c \approx \frac{1}{10\,\tau_{\mathrm{rms}}}
$$

The unit of bandwidth is Hertz (Hz). If the signal bandwidth $B \ll B_c$, then there is flat fading and no frequency selectivity. If $B \gtrsim B_c$, then there is frequency-selective fading, and different OFDM subcarriers see different gains/phases, and inter-symbol interference (ISI) can occur for single-carrier systems.

For example, if a channel has 20 MHz of bandwidth, instead of sending one wide 20 MHz signal, OFDM might divide it into 64 subcarriers, each 312.5 kHz wide. Each subcarrier carries part of the data in parallel, which enables high data rates, efficient use of spectrum, and resilience to multipath fading.


#### 2.3 FFT

Let $X[k]$ be subcarrier symbols for $k=0,\dots,N-1$. A baseband OFDM time-domain block is an IFFT (meaning, the discrete Fast Fourier Transform can be used to efficiently create the OFDM signal):

$$
x[n] = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} X[k] e^{j 2\pi kn/N}, \quad n=0,\dots,N-1
$$

In this equation, the term $e^{j 2\pi kn / N}$ is a complex exponential (a rotating phasor combining sine and cosine) that serves as the basis function in the inverse Fourier transform; it encodes sinusoidal oscillations at different frequencies, and the weighted sum of these exponentials with coefficients $X[k]$ reconstructs the time-domain signal $x[n]$.

Today, there are many efficient digital implementations of the FFT, and designing hardware to perform the inverse FFT is a well-established practice. This ease of implementation is another key reason why OFDM has been widely adopted in standardized consumer technologies.


#### 2.4 Cyclic Prefix (CP) in OFDM

As we know by now, in wireless communication, one of the main challenges is multipath propagation. The transmitted signal often reaches the receiver over several paths with different delays, which leads to intersymbol interference (ISI), where the tail of one OFDM symbol overlaps with the next, and intercarrier interference (ICI), where the orthogonality between subcarriers is destroyed.

The cyclic prefix $CP$ is a guard interval inserted at the beginning of every OFDM symbol to mitigate these effects. It is constructed by copying the last $N_{CP}$ samples of the OFDM symbol and prefixing them to the front, so that an OFDM symbol of length $N$ becomes $N+N_{CP}$ samples long.

In practice, the cyclic prefix must be chosen such that its duration in time is at least as long as the maximum expected multipath delay spread of the channel. A longer CP provides robustness but reduces efficiency, while a shorter CP improves efficiency but risks losing orthogonality in severe multipath environments.

In the time domain, the cyclic prefix acts as a buffer. As long as the entire channel impulse response fits within the CP, the multipath-delayed copies of the signal only overlap with the prefix and do not interfere with the useful part of the symbol. Orthogonality between subcarriers is preserved.

This mechanism has trade-offs. If the cyclic prefix is long enough, specifically if its length in samples $N_{CP}$ is at least as large as the effective length of the channel impulse response minus one, then both ISI and ICI are eliminated. However, the prefix does not carry new information, so it reduces efficiency. The useful data occupies only $N$ samples out of the total $N+N_{CP}$, and the spectral efficiency is therefore reduced to $N/(N+N_{CP})$. For example, with a prefix of one sixteenth of the symbol length, the resulting efficiency is only about 94 percent.

#### 2.5 Example



In [ ]:
# --- OFDM helper functions ---
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft, ifft

np.random.seed(42)

def qpsk_mod(bits):
    """Gray-coded QPSK: (bQ,bI) -> {00:+1+1j, 01:-1+1j, 11:-1-1j, 10:+1-1j}/sqrt(2)"""
    bits = np.asarray(bits).reshape(-1, 2)
    mapping = {(0,0): 1+1j, (0,1): -1+1j, (1,1): -1-1j, (1,0): 1-1j}
    syms = np.array([mapping[tuple(b)] for b in bits], dtype=complex) / np.sqrt(2)
    return syms

def qpsk_demod(syms):
    """Inverse of mapping above: return bits as (bQ,bI)."""
    syms = np.asarray(syms)
    bQ = (syms.imag < 0).astype(np.int8)  # 0 if +Q, 1 if -Q
    bI = (syms.real < 0).astype(np.int8)  # 0 if +I, 1 if -I
    return np.column_stack([bQ, bI]).ravel()

def ber(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    return np.mean(a != b)

def ofdm_mod(X, cp_len):
    """X: (num_symbols, N) -> time-domain with CP (1D)."""
    X = np.atleast_2d(X)
    num_sym, N = X.shape
    x_blocks = ifft(X, axis=1) * np.sqrt(N)
    with_cp = np.concatenate([x_blocks[:, -cp_len:], x_blocks], axis=1)
    return with_cp.reshape(-1)

def ofdm_demod(rx, N, cp_len):
    """rx 1D -> X_hat (num_symbols, N), assumes length multiple of N+cp_len."""
    sym_len = N + cp_len
    assert len(rx) % sym_len == 0
    num_sym = len(rx) // sym_len
    blocks = rx.reshape(num_sym, sym_len)[:, cp_len:]
    X_hat = fft(blocks, axis=1) / np.sqrt(N)
    return X_hat

def add_awgn(x, snr_db):
    sig_pow = np.mean(np.abs(x)**2)
    snr_lin = 10**(snr_db/10)
    n_pow = sig_pow / snr_lin
    n = np.sqrt(n_pow/2) * (np.random.randn(*x.shape) + 1j*np.random.randn(*x.shape))
    return x + n

def channel_freq_response(h, N):
    L = len(h)
    if L < N:
        h_pad = np.pad(h, (0, N-L))
    else:
        h_pad = h[:N]
    return fft(h_pad)

def rayleigh_multipath(delays, pwr_db):
    """Discrete-tap channel; taps at integer delays."""
    L = max(delays) + 1
    h = np.zeros(L, dtype=complex)
    pwr = 10**(np.array(pwr_db)/10)
    for d, p in zip(delays, pwr):
        mag = np.sqrt(p/2) * np.sqrt(np.random.chisquare(2))   # Rayleigh magnitude
        ph  = np.exp(1j * 2*np.pi*np.random.rand())            # random phase
        h[d] = mag * ph
    # Normalize average power to 1
    norm = np.sqrt(np.sum(np.abs(h)**2))
    return h / (norm + 1e-12)

In [ ]:
# --- OFDM Estimation over Multipath ---
N        = 64      # number of OFDM subcarriers
snr_db   = 100     # signal-to-noise ratio [dB]
num_sym  = 500     # number of OFDM symbols
cp_len   = round(N/8)  # cyclic prefix length

# Example 5-tap Power Delay Profile (PDP)
delays = [0, 3, 5, 9, 12]   # delay in samples
pwr_db = [1, 1, 1, 1, 1]    # average tap power [dB]

# Build and visualize channel
h = rayleigh_multipath(delays, pwr_db)
H = channel_freq_response(h, N)
print(f"Channel length L={len(h)}, CP={cp_len} (need ≥ {len(h)-1} for no ISI)")

# Impulse Response
fig, ax = plt.subplots(figsize=(10, 3))
markerline, stemlines, baseline = ax.stem(np.arange(len(h)), np.abs(h), basefmt=" ")
plt.setp(markerline, color="gold", markersize=6)
plt.setp(stemlines, color="lime", linewidth=1.2)
ax.set_facecolor((0.15, 0.15, 0.22))
ax.grid(True, alpha=0.3)
ax.set_title("Impulse Response |h[n]|")
ax.set_xlabel("Tap index n")
ax.set_ylabel("Magnitude")
plt.tight_layout()
plt.show()

# Frequency Response
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.abs(H), color="cyan", linewidth=2)
ax.set_facecolor((0.15, 0.15, 0.22))
ax.grid(True, alpha=0.3)
ax.set_title("Frequency Response |H[k]|")
ax.set_xlabel("Subcarrier index k")
ax.set_ylabel("Magnitude")
plt.tight_layout()
plt.show()

#  OFDM Transmission / Reception
bits = np.random.randint(0, 2, size=num_sym * 2 * N)
X = qpsk_mod(bits).reshape(num_sym, N)
x = ofdm_mod(X, cp_len)

# Channel and noise
y = np.convolve(x, h, mode='full')
y = add_awgn(y, snr_db)

# Trim and demodulate
sym_len = N + cp_len
y = y[:num_sym * sym_len]
Y = ofdm_demod(y, N, cp_len)

# One-tap equalization (perfect CSI)
H = channel_freq_response(h, N)
Y_eq = Y / (H + 1e-12)

syms_hat = Y_eq.reshape(-1)
bits_hat = qpsk_demod(syms_hat)
ber_good = ber(bits, bits_hat)

# Constellation Plot 
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(syms_hat.real[:6000], syms_hat.imag[:6000],
           s=6, alpha=0.6, color='lime')
ax.set_facecolor((0.15, 0.15, 0.22))
ax.grid(True, alpha=0.3)
ax.set_title("Equalized Constellation (Adequate CP)")
ax.set_xlabel("In-Phase (I)")
ax.set_ylabel("Quadrature (Q)")
ax.axis('equal')
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

print(f"SNR = {snr_db} dB")
print(f"Number of OFDM Subcarriers N = {N}")
print(f"Resulting Bit Error Rate (BER) = {ber_good * 100:.4f}%")

In [ ]:
# --- OFDM Subcarrier Spectra Visualization ---
def plot_ofdm_subcarriers(N, dense=8001, extra=0.1, show_grid=True):

    import numpy as np
    import matplotlib.pyplot as plt

    # Extended frequency axis
    fmin, fmax = -0.5 * (1 + extra), 0.5 * (1 + extra)
    f = np.linspace(fmin, fmax, dense, endpoint=True)

    # Subcarrier center frequencies (DC-centered)
    k = np.arange(N)
    f0 = (k - N/2) / N

    # Magnitude of sinc spectra for all subcarriers
    df = f[:, None] - f0[None, :]
    with np.errstate(divide='ignore', invalid='ignore'):
        mag = np.abs(np.sinc(N * df) / np.sinc(df))
    mag[np.isnan(mag)] = 1.0

    # Plot
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.set_facecolor((0.12, 0.12, 0.18))

    # Plot all subcarrier shapes faintly
    ax.plot(f, mag, linewidth=0.6, alpha=0.25, color='cyan')

    # Optional vertical grid lines
    if show_grid:
        ax.vlines(f0, ymin=0, ymax=1.02, linewidth=0.5, linestyles='dotted', alpha=0.15, color='white')

    # Shade out-of-band regions
    ax.axvspan(fmin, -0.5, color='gray', alpha=0.25, label="Out of band")
    ax.axvspan(0.5, fmax, color='gray', alpha=0.25)

    # Nyquist edges
    ax.axvline(-0.5, color='red', linestyle='--', linewidth=1.0, label="Nyquist edges")
    ax.axvline(+0.5, color='red', linestyle='--', linewidth=1.0)

    # Styling
    ax.set_xlim(fmin, fmax)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"OFDM Subcarriers in Frequency (N={N})")
    ax.set_xlabel("Normalized frequency (cycles/sample)")
    ax.set_ylabel("Magnitude")
    ax.grid(True, alpha=0.25)
    ax.legend(facecolor=(0.2, 0.2, 0.3), edgecolor='white', labelcolor='white', loc='upper right')
    plt.tight_layout()
    plt.show()

plot_ofdm_subcarriers(N, extra=0.1)